# 08 — Análise de Recursos Operacionais

**Objetivo:** Mapear para ONDE vão as horas operacionais hoje — por canal, por assunto, por par canal×assunto — e classificar cada par no diagnóstico de 6 cenários para identificar desperdício e oportunidades de realocação.

**Dados:** `customer_support_tickets.csv` — 8.469 tickets, 2.769 fechados com CSAT.  
**Métrica-chave:** `duration_hours = abs(TTR - FRT)` em horas.

---

## 1. Carga de Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import squarify
import warnings
warnings.filterwarnings('ignore')

# Plotly para Sankey
try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
    print("Plotly disponivel para Sankey")
except ImportError:
    HAS_PLOTLY = False
    print("Plotly nao disponivel — Sankey sera omitido")

csv_path = 'data/customer_support_tickets.csv'
df_raw = pd.read_csv(csv_path)
print(f'Dataset total: {len(df_raw)} tickets')

# Filtrar fechados com CSAT
df = df_raw[df_raw['Ticket Status'] == 'Closed'].copy()
df['First Response Time'] = pd.to_datetime(df['First Response Time'], errors='coerce')
df['Time to Resolution'] = pd.to_datetime(df['Time to Resolution'], errors='coerce')
df['duration_hours'] = (df['Time to Resolution'] - df['First Response Time']).dt.total_seconds().abs() / 3600
df = df.dropna(subset=['Customer Satisfaction Rating', 'duration_hours'])

df = df.rename(columns={
    'Ticket Channel': 'channel',
    'Ticket Subject': 'subject',
    'Customer Satisfaction Rating': 'csat',
})

print(f'Tickets fechados com CSAT e duracao: {len(df)}')
print(f'Horas totais operacionais: {df["duration_hours"].sum():,.0f}h')
print(f'Duracao media: {df["duration_hours"].mean():.1f}h | mediana: {df["duration_hours"].median():.1f}h')
print(f'CSAT medio: {df["csat"].mean():.2f}')
print(f'\nCanais: {sorted(df["channel"].unique())}')
print(f'Assuntos: {sorted(df["subject"].unique())}')

## 2. Horas por Canal

Quanto tempo operacional cada canal consome? Barras horizontais ordenadas por total de horas.

In [ ]:
hours_by_channel = df.groupby('channel')['duration_hours'].agg(['sum', 'mean', 'count']).sort_values('sum', ascending=True)
hours_by_channel.columns = ['total_hours', 'avg_hours', 'n_tickets']
total = hours_by_channel['total_hours'].sum()

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(hours_by_channel.index, hours_by_channel['total_hours'], color='#3b82f6', edgecolor='white')
for bar, (_, row) in zip(bars, hours_by_channel.iterrows()):
    pct = row['total_hours'] / total * 100
    ax.text(bar.get_width() + total * 0.01, bar.get_y() + bar.get_height()/2,
            f'{row["total_hours"]:,.0f}h ({pct:.1f}%) — {row["n_tickets"]} tickets, média {row["avg_hours"]:.0f}h',
            va='center', fontsize=10)
ax.set_xlabel('Total de Horas Operacionais')
ax.set_title('Distribuição de Horas Operacionais por Canal')
ax.set_xlim(0, total * 1.35)
plt.tight_layout()
plt.savefig('process-log/screenshots/p6_hours_by_channel.png', dpi=150, bbox_inches='tight')
plt.show()

print('Horas por canal:')
for ch, row in hours_by_channel.sort_values('total_hours', ascending=False).iterrows():
    print(f'  {ch}: {row["total_hours"]:,.0f}h ({row["total_hours"]/total*100:.1f}%) — {row["n_tickets"]:.0f} tickets')

## 3. Horas por Assunto

Quais assuntos consomem mais horas? Ordenado por total decrescente.

In [ ]:
hours_by_subject = df.groupby('subject')['duration_hours'].agg(['sum', 'mean', 'count']).sort_values('sum', ascending=False)
hours_by_subject.columns = ['total_hours', 'avg_hours', 'n_tickets']

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(hours_by_subject.index[::-1], hours_by_subject['total_hours'][::-1], color='#8b5cf6', edgecolor='white')
for bar, (subj, row) in zip(bars, hours_by_subject[::-1].iterrows()):
    pct = row['total_hours'] / total * 100
    ax.text(bar.get_width() + total * 0.005, bar.get_y() + bar.get_height()/2,
            f'{row["total_hours"]:,.0f}h ({pct:.1f}%)', va='center', fontsize=9)
ax.set_xlabel('Total de Horas Operacionais')
ax.set_title('Distribuição de Horas Operacionais por Assunto')
plt.tight_layout()
plt.savefig('process-log/screenshots/p6_hours_by_subject.png', dpi=150, bbox_inches='tight')
plt.show()

print('Horas por assunto:')
for subj, row in hours_by_subject.iterrows():
    print(f'  {subj}: {row["total_hours"]:,.0f}h ({row["total_hours"]/total*100:.1f}%) — {row["n_tickets"]:.0f} tickets, média {row["avg_hours"]:.0f}h')

## 4. Horas por Par Canal × Assunto

Heatmap anotado (horas) + treemap (tamanho = horas, cor = CSAT médio). Identifica os pares que mais consomem recursos e sua qualidade de atendimento.

In [ ]:
pair_stats = df.groupby(['channel', 'subject']).agg(
    total_hours=('duration_hours', 'sum'),
    avg_csat=('csat', 'mean'),
    n_tickets=('duration_hours', 'count')
).reset_index()

# Heatmap de horas
pivot_hours = pair_stats.pivot(index='subject', columns='channel', values='total_hours').fillna(0)
pivot_csat = pair_stats.pivot(index='subject', columns='channel', values='avg_csat')

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Heatmap horas
sns.heatmap(pivot_hours, annot=True, fmt=',.0f', cmap='YlOrRd', ax=axes[0],
            linewidths=0.5, cbar_kws={'label': 'Horas'})
axes[0].set_title('Horas Operacionais por Par Canal × Assunto')
axes[0].set_xlabel('Canal')
axes[0].set_ylabel('Assunto')

# Heatmap CSAT
sns.heatmap(pivot_csat, annot=True, fmt='.2f', cmap='RdYlGn', ax=axes[1],
            linewidths=0.5, vmin=1, vmax=5, cbar_kws={'label': 'CSAT Médio'})
axes[1].set_title('CSAT Médio por Par Canal × Assunto')
axes[1].set_xlabel('Canal')
axes[1].set_ylabel('Assunto')

plt.tight_layout()
plt.savefig('process-log/screenshots/p6_heatmap_hours_csat.png', dpi=150, bbox_inches='tight')
plt.show()

# Treemap: tamanho = horas, cor = CSAT
pair_stats_sorted = pair_stats.sort_values('total_hours', ascending=False)
labels = [f'{r.channel}\n{r.subject}\n{r.total_hours:,.0f}h\nCSAT {r.avg_csat:.1f}'
          for _, r in pair_stats_sorted.iterrows()]

# Normalizar CSAT para colormap
norm = plt.Normalize(vmin=pair_stats_sorted['avg_csat'].min(), vmax=pair_stats_sorted['avg_csat'].max())
cmap = plt.cm.RdYlGn
colors = [cmap(norm(v)) for v in pair_stats_sorted['avg_csat']]

fig, ax = plt.subplots(figsize=(18, 12))
squarify.plot(sizes=pair_stats_sorted['total_hours'].values,
              label=labels, color=colors, alpha=0.85, ax=ax,
              text_kwargs={'fontsize': 7, 'wrap': True})
ax.set_title('Treemap: Horas Operacionais (tamanho) × CSAT (cor verde=alto, vermelho=baixo)', fontsize=14)
ax.axis('off')
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label='CSAT Médio', shrink=0.6)
plt.tight_layout()
plt.savefig('process-log/screenshots/p6_treemap_hours_csat.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Top 10 pares por horas:')
for _, r in pair_stats_sorted.head(10).iterrows():
    print(f'  {r.channel} × {r.subject}: {r.total_hours:,.0f}h ({r.n_tickets} tickets, CSAT {r.avg_csat:.2f})')

## 5. Classificação por Cenário (6 cenários)

Lógica de classificação para cada par canal×assunto (mín. 5 tickets):

| Cenário | Condição |
|---------|----------|
| **Acelerar** | Pearson r(duração, CSAT) < -0.3 → mais tempo = pior CSAT |
| **Desacelerar** | Pearson r > 0.3 → mais tempo = melhor CSAT |
| **Redirecionar** | CSAT gap > 0.5 vs melhor canal + redirecionamento viável |
| **Quarentena** | Gap > 0.5 mas sem alvo viável, ou todos os canais ruins |
| **Manter** | CSAT ≥ 3.5, sem correlação significativa |
| **Liberar** | Nenhuma das anteriores → recursos sem retorno claro |

Regras de viabilidade de redirecionamento:
- Email ↔ Phone ↔ Chat: todos bidirecionais OK
- Social media → Email/Phone: OK | Social media → Chat: NÃO | Qualquer → Social media: NÃO

In [ ]:
# Matriz de viabilidade de redirecionamento
REDIRECT_VIABLE = {
    'Email': {'Phone': True, 'Chat': True, 'Social media': False},
    'Chat': {'Phone': True, 'Email': True, 'Social media': False},
    'Phone': {'Email': True, 'Chat': True, 'Social media': False},
    'Social media': {'Email': True, 'Phone': True, 'Chat': False},
}

channels = sorted(df['channel'].unique())
subjects = sorted(df['subject'].unique())

# Pre-computar CSAT medio por assunto-canal (min 5 tickets)
subj_ch_csat = {}
for sub in subjects:
    ch_stats = []
    for ch in channels:
        subset = df[(df['channel'] == ch) & (df['subject'] == sub)]
        if len(subset) >= 5:
            ch_stats.append({'channel': ch, 'avg_csat': round(subset['csat'].mean(), 2)})
    subj_ch_csat[sub] = sorted(ch_stats, key=lambda x: -x['avg_csat'])

# Classificar cada par
diags = []
for ch in channels:
    for sub in subjects:
        subset = df[(df['channel'] == ch) & (df['subject'] == sub)]
        if len(subset) < 5:
            continue
        
        r_val, _ = pearsonr(subset['duration_hours'], subset['csat'])
        r_val = round(r_val, 3)
        pair_csat = round(subset['csat'].mean(), 2)
        pair_hours = subset['duration_hours'].sum()
        n = len(subset)
        
        chs = subj_ch_csat.get(sub, [])
        best_ch = chs[0] if chs else None
        
        if r_val < -0.3:
            scenario = 'acelerar'
        elif r_val > 0.3:
            scenario = 'desacelerar'
        else:
            gap = best_ch['avg_csat'] - pair_csat if best_ch else 0
            if gap > 0.5 and best_ch and best_ch['channel'] != ch:
                # Verificar se existe canal viavel para redirecionar
                viable = [c for c in chs
                          if c['channel'] != ch
                          and c['avg_csat'] > pair_csat + 0.3
                          and REDIRECT_VIABLE.get(ch, {}).get(c['channel'], False)]
                scenario = 'redirecionar' if viable else 'quarentena'
            elif pair_csat >= 3.5:
                scenario = 'manter'
            else:
                all_bad = all(c['avg_csat'] < 3.0 for c in chs)
                scenario = 'quarentena' if (all_bad and pair_csat < 3.0) else 'liberar'
        
        diags.append({
            'channel': ch, 'subject': sub, 'scenario': scenario,
            'r_pair': r_val, 'pair_csat': pair_csat,
            'total_hours': round(pair_hours, 1), 'n_tickets': n
        })

diag_df = pd.DataFrame(diags)
print(f'Total de pares classificados: {len(diag_df)}')
print(f'Horas classificadas: {diag_df["total_hours"].sum():,.0f}h de {total:,.0f}h total')
print(f'\nDistribuicao de cenarios:')
scenario_counts = diag_df['scenario'].value_counts()
for sc, cnt in scenario_counts.items():
    sc_hours = diag_df[diag_df['scenario'] == sc]['total_hours'].sum()
    print(f'  {sc}: {cnt} pares, {sc_hours:,.0f}h ({sc_hours/total*100:.1f}%)')

# Scatter: r vs CSAT colorido por cenario
SC_COLORS = {
    'acelerar': '#ef4444', 'desacelerar': '#3b82f6', 'redirecionar': '#f97316',
    'quarentena': '#eab308', 'manter': '#22c55e', 'liberar': '#9ca3af'
}

fig, ax = plt.subplots(figsize=(14, 8))
for sc, col in SC_COLORS.items():
    sd = diag_df[diag_df['scenario'] == sc]
    if len(sd) == 0:
        continue
    ax.scatter(sd['r_pair'], sd['pair_csat'], c=col, label=f'{sc} ({len(sd)})',
               s=sd['total_hours'] / diag_df['total_hours'].max() * 500 + 30,
               alpha=0.7, edgecolors='black', linewidths=0.5)
ax.axvline(x=-0.3, color='red', linestyle=':', alpha=0.5, label='r = -0.3')
ax.axvline(x=0.3, color='blue', linestyle=':', alpha=0.5, label='r = 0.3')
ax.axhline(y=3.5, color='green', linestyle=':', alpha=0.5, label='CSAT = 3.5')
ax.set_xlabel('Pearson r (duração × CSAT)')
ax.set_ylabel('CSAT Médio do Par')
ax.set_title('Mapa de Cenários: r vs CSAT (tamanho = horas)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('process-log/screenshots/p6_scenario_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Recursos por Cenário

Stacked bar chart: horas por cenário, colorido por canal. Tabela resumo com % do total.

In [ ]:
# Stacked bar: horas por cenario, colorido por canal
scenario_channel = diag_df.groupby(['scenario', 'channel'])['total_hours'].sum().unstack(fill_value=0)
scenario_order = ['acelerar', 'desacelerar', 'redirecionar', 'quarentena', 'manter', 'liberar']
scenario_channel = scenario_channel.reindex([s for s in scenario_order if s in scenario_channel.index])

CH_COLORS = {'Chat': '#3b82f6', 'Email': '#10b981', 'Phone': '#f97316', 'Social media': '#8b5cf6'}

fig, ax = plt.subplots(figsize=(14, 7))
scenario_channel.plot(kind='barh', stacked=True, ax=ax,
                      color=[CH_COLORS.get(c, '#9ca3af') for c in scenario_channel.columns],
                      edgecolor='white', linewidth=0.5)
for i, (sc, row) in enumerate(scenario_channel.iterrows()):
    total_sc = row.sum()
    pct = total_sc / total * 100
    ax.text(total_sc + total * 0.005, i, f'{total_sc:,.0f}h ({pct:.1f}%)', va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Horas Operacionais')
ax.set_title('Horas Operacionais por Cenário (colorido por Canal)')
ax.legend(title='Canal', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('process-log/screenshots/p6_hours_by_scenario_stacked.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabela resumo
print('=' * 70)
print('TABELA RESUMO: RECURSOS POR CENÁRIO')
print('=' * 70)
print(f'{"Cenário":<15} {"Pares":>6} {"Tickets":>8} {"Horas":>10} {"% Total":>8} {"CSAT Méd":>9}')
print('-' * 70)
total_hours_all = diag_df['total_hours'].sum()
for sc in scenario_order:
    sd = diag_df[diag_df['scenario'] == sc]
    if len(sd) == 0:
        continue
    h = sd['total_hours'].sum()
    # Weighted average CSAT by hours
    w_csat = np.average(sd['pair_csat'], weights=sd['total_hours']) if h > 0 else 0
    print(f'{sc:<15} {len(sd):>6} {sd["n_tickets"].sum():>8} {h:>10,.0f} {h/total_hours_all*100:>7.1f}% {w_csat:>9.2f}')
print('-' * 70)
print(f'{"TOTAL":<15} {len(diag_df):>6} {diag_df["n_tickets"].sum():>8} {total_hours_all:>10,.0f} {100:>7.1f}%')

# Metrica-chave: % liberar
liberar_hours = diag_df[diag_df['scenario'] == 'liberar']['total_hours'].sum()
liberar_pct = liberar_hours / total_hours_all * 100
print(f'\n>>> MÉTRICA-CHAVE: Horas "liberar" = {liberar_hours:,.0f}h = {liberar_pct:.1f}% do total <<<')
print(f'>>> Essas são horas sem retorno claro em qualidade — candidatas a realocação <<<')

## 7. Análise de Desperdício

Cenários de realocação: se movermos 25%, 50% ou 75% das horas de "liberar" para pares "acelerar", qual o impacto potencial?

In [ ]:
acelerar_hours = diag_df[diag_df['scenario'] == 'acelerar']['total_hours'].sum()
acelerar_csat = np.average(
    diag_df[diag_df['scenario'] == 'acelerar']['pair_csat'],
    weights=diag_df[diag_df['scenario'] == 'acelerar']['total_hours']
) if acelerar_hours > 0 else 0

manter_hours = diag_df[diag_df['scenario'] == 'manter']['total_hours'].sum()
manter_csat = np.average(
    diag_df[diag_df['scenario'] == 'manter']['pair_csat'],
    weights=diag_df[diag_df['scenario'] == 'manter']['total_hours']
) if manter_hours > 0 else 0

print('=' * 70)
print('ANÁLISE DE DESPERDÍCIO E CENÁRIOS DE REALOCAÇÃO')
print('=' * 70)
print(f'Horas "liberar" (desperdício): {liberar_hours:,.0f}h ({liberar_pct:.1f}%)')
print(f'Horas "acelerar" (demanda): {acelerar_hours:,.0f}h (CSAT médio: {acelerar_csat:.2f})')
print(f'Horas "manter" (eficientes): {manter_hours:,.0f}h (CSAT médio: {manter_csat:.2f})')
print()

scenarios = [0.25, 0.50, 0.75]
results = []
for pct in scenarios:
    moved = liberar_hours * pct
    new_acelerar = acelerar_hours + moved
    remaining_liberar = liberar_hours * (1 - pct)
    # Estimativa conservadora: horas realocadas para acelerar reduzem tempo medio
    # => potencial melhoria de CSAT nos pares acelerar
    boost_ratio = moved / acelerar_hours if acelerar_hours > 0 else 0
    results.append({
        'cenario': f'{int(pct*100)}% realocado',
        'horas_movidas': moved,
        'novo_acelerar': new_acelerar,
        'liberar_restante': remaining_liberar,
        'boost_ratio': boost_ratio,
    })
    print(f'Cenário {int(pct*100)}%: mover {moved:,.0f}h de liberar → acelerar')
    print(f'  Acelerar passa de {acelerar_hours:,.0f}h → {new_acelerar:,.0f}h (+{boost_ratio*100:.0f}% capacidade)')
    print(f'  Liberar restante: {remaining_liberar:,.0f}h')
    print()

# Visualizacao
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart: cenarios
labels = ['Atual'] + [r['cenario'] for r in results]
acelerar_vals = [acelerar_hours] + [r['novo_acelerar'] for r in results]
liberar_vals = [liberar_hours] + [r['liberar_restante'] for r in results]
outros = total_hours_all - acelerar_hours - liberar_hours

x = np.arange(len(labels))
width = 0.3
axes[0].bar(x - width, acelerar_vals, width, label='Acelerar', color='#ef4444')
axes[0].bar(x, liberar_vals, width, label='Liberar (desperdício)', color='#9ca3af')
axes[0].bar(x + width, [outros] * len(labels), width, label='Outros cenários', color='#3b82f6', alpha=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=15)
axes[0].set_ylabel('Horas')
axes[0].set_title('Cenários de Realocação de Recursos')
axes[0].legend()

# Pie chart: distribuicao atual
sc_hours = diag_df.groupby('scenario')['total_hours'].sum()
sc_hours = sc_hours.reindex([s for s in scenario_order if s in sc_hours.index])
colors_pie = [SC_COLORS.get(s, '#9ca3af') for s in sc_hours.index]
axes[1].pie(sc_hours.values, labels=[f'{s}\n{h:,.0f}h' for s, h in zip(sc_hours.index, sc_hours.values)],
            colors=colors_pie, autopct='%1.1f%%', startangle=90, pctdistance=0.85)
axes[1].set_title('Distribuição Atual de Horas por Cenário')

plt.tight_layout()
plt.savefig('process-log/screenshots/p6_reallocation_scenarios.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Diagrama Sankey: Canal → Cenário → Ação

Fluxo visual: de qual canal vêm as horas, para qual cenário são classificadas, e qual ação recomendada.

In [ ]:
if HAS_PLOTLY:
    # Mapear cenario -> acao
    SCENARIO_ACTION = {
        'acelerar': 'Investir + Automatizar',
        'desacelerar': 'Manter Ritmo Atual',
        'redirecionar': 'Mover para Outro Canal',
        'quarentena': 'Investigar / Pausar',
        'manter': 'Manter Como Está',
        'liberar': 'Realocar Recursos',
    }
    
    # Nodes: channels + scenarios + actions
    ch_list = sorted(diag_df['channel'].unique())
    sc_list = [s for s in scenario_order if s in diag_df['scenario'].unique()]
    action_list = list(dict.fromkeys([SCENARIO_ACTION[s] for s in sc_list]))
    
    all_nodes = ch_list + sc_list + action_list
    node_idx = {n: i for i, n in enumerate(all_nodes)}
    
    # Node colors
    node_colors = []
    for n in all_nodes:
        if n in CH_COLORS:
            node_colors.append(CH_COLORS[n])
        elif n in SC_COLORS:
            node_colors.append(SC_COLORS[n])
        else:
            node_colors.append('#6b7280')
    
    sources, targets, values, link_colors = [], [], [], []
    
    # Channel -> Scenario links
    for _, row in diag_df.groupby(['channel', 'scenario'])['total_hours'].sum().reset_index().iterrows():
        sources.append(node_idx[row['channel']])
        targets.append(node_idx[row['scenario']])
        values.append(row['total_hours'])
        link_colors.append(SC_COLORS.get(row['scenario'], '#9ca3af'))
    
    # Scenario -> Action links
    for sc in sc_list:
        action = SCENARIO_ACTION[sc]
        sc_h = diag_df[diag_df['scenario'] == sc]['total_hours'].sum()
        sources.append(node_idx[sc])
        targets.append(node_idx[action])
        values.append(sc_h)
        link_colors.append(SC_COLORS.get(sc, '#9ca3af'))
    
    # Make link colors semi-transparent
    link_colors_rgba = []
    for c in link_colors:
        r, g, b = int(c[1:3], 16), int(c[3:5], 16), int(c[5:7], 16)
        link_colors_rgba.append(f'rgba({r},{g},{b},0.4)')
    
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15, thickness=20,
            label=[f'{n}\n{diag_df[diag_df["channel"]==n]["total_hours"].sum():,.0f}h' if n in ch_list
                   else f'{n}\n{diag_df[diag_df["scenario"]==n]["total_hours"].sum():,.0f}h' if n in sc_list
                   else n
                   for n in all_nodes],
            color=node_colors,
        ),
        link=dict(
            source=sources, target=targets, value=values,
            color=link_colors_rgba,
        )
    )])
    fig.update_layout(
        title_text='Fluxo de Recursos: Canal → Cenário → Ação Recomendada',
        font_size=11, width=1200, height=600
    )
    fig.write_image('process-log/screenshots/p6_sankey_resource_flow.png', scale=2)
    fig.show()
    print('Sankey salvo em process-log/screenshots/p6_sankey_resource_flow.png')
else:
    print('Plotly nao disponivel — Sankey omitido.')
    print('Para gerar: pip install plotly kaleido')

## 9. Conclusões

Síntese dos achados e recomendações operacionais.

In [ ]:
print('=' * 70)
print('CONCLUSÕES — ANÁLISE DE RECURSOS OPERACIONAIS')
print('=' * 70)

print(f'''
1. VOLUME TOTAL: {total_hours_all:,.0f} horas operacionais em {len(diag_df)} pares canal×assunto
   ({diag_df["n_tickets"].sum()} tickets fechados com CSAT)

2. DISTRIBUIÇÃO POR CENÁRIO:''')
for sc in scenario_order:
    sd = diag_df[diag_df['scenario'] == sc]
    if len(sd) == 0:
        continue
    h = sd['total_hours'].sum()
    print(f'   - {sc.upper()}: {h:,.0f}h ({h/total_hours_all*100:.1f}%) — {len(sd)} pares')

print(f'''
3. DESPERDÍCIO IDENTIFICADO:
   - Horas "liberar": {liberar_hours:,.0f}h ({liberar_pct:.1f}%) — recursos sem retorno claro em CSAT
   - São pares onde duração não correlaciona com satisfação e CSAT < 3.5
   - Candidatas imediatas a realocação ou automação

4. OPORTUNIDADE DE REALOCAÇÃO:
   - Cenário conservador (25%): mover {liberar_hours*0.25:,.0f}h para acelerar
   - Cenário moderado (50%): mover {liberar_hours*0.50:,.0f}h para acelerar
   - Cenário agressivo (75%): mover {liberar_hours*0.75:,.0f}h para acelerar

5. RECOMENDAÇÕES:
   a) ACELERAR: investir em automação e SLA mais agressivo nos pares identificados
   b) LIBERAR: reduzir esforço manual, implementar autoatendimento/chatbot
   c) REDIRECIONAR: mover tickets dos canais ineficientes para canais com melhor CSAT
   d) MANTER: preservar qualidade nos pares que já funcionam bem
''')
print('=' * 70)

# Salvar diagnostico completo
diag_df.to_csv('data/resource_analysis_diagnostics.csv', index=False)
print('Diagnóstico salvo em data/resource_analysis_diagnostics.csv')